In [90]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models import efficientnet_b0
from PIL import Image
import pandas as pd
import numpy as np
import pydicom
from sklearn.metrics import accuracy_score

In [91]:
csv_path = "../tocsv/combined_labels.csv"  
ct_folder = "../data/stage_2_train_images"  

In [92]:
df = pd.read_csv(csv_path)

# Build full image paths safely
def get_full_path(row):
    if row['modality'].upper() == 'CT':
        path = os.path.join(ct_folder, row['image_path'] + ".dcm")
        return path if os.path.exists(path) else None
    else:
        return row['image_path'] if os.path.exists(row['image_path']) else None

df['image_path'] = df.apply(get_full_path, axis=1)
df = df.dropna(subset=['image_path']).reset_index(drop=True)



In [93]:
import os
import pydicom
import numpy as np
from PIL import Image
from torch.utils.data import Dataset

class ICHDataset(Dataset):
    def __init__(self, image_dir, labels_df, transform=None):
        self.image_dir = image_dir
        self.labels_df = labels_df
        self.transform = transform

        # create map {image_id : full_path}
        self.image_map = {}
        for root, _, files in os.walk(self.image_dir):
            for file in files:
                image_id = os.path.splitext(file)[0]
                self.image_map[image_id] = os.path.join(root, file)

    def __len__(self):
        return len(self.labels_df)

    def __getitem__(self, idx):
        row = self.labels_df.iloc[idx]
        img_name = str(row['image_path']).split('.')[0]

        if img_name not in self.image_map:
            raise FileNotFoundError(f"Image {img_name} not found in {self.image_dir}")

        img_path = self.image_map[img_name]

        # Handle DICOM (CT) and normal images (MRI)
        if img_path.lower().endswith('.dcm'):
            dicom_data = pydicom.dcmread(img_path)
            image = dicom_data.pixel_array.astype(np.float32)

            # Normalize and convert to PIL Image
            image = (image - np.min(image)) / (np.max(image) - np.min(image))
            image = (image * 255).astype(np.uint8)
            image = Image.fromarray(image).convert("RGB")
        else:
            image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        # Labels (convert to tensor later in DataLoader)
        labels = row[['epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']].values.astype(np.float32)

        # Detect modality
        modality = 0 if 'dcm' in img_path.lower() else 1  # 0 = CT, 1 = MRI

        return image, modality, labels


In [94]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [95]:
import torch
import torch.nn as nn
from torchvision import models

class ResNet50Combined(nn.Module):
    def __init__(self, num_classes=5):
        super(ResNet50Combined, self).__init__()
        self.base_model = models.resnet50(pretrained=True)
        in_features = self.base_model.fc.in_features
        self.base_model.fc = nn.Identity()  # remove default FC layer

        # Modality embedding: 2 -> 128  (CT=0, MRI=1)
        self.modality_embed = nn.Embedding(2, 128)

        # Combined classifier
        self.classifier = nn.Sequential(
            nn.Linear(in_features + 128, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
            nn.Sigmoid()
        )

    def forward(self, x, modality):
        x = self.base_model(x)
        m = self.modality_embed(modality)
        combined = torch.cat((x, m), dim=1)
        return self.classifier(combined)


In [96]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet50Combined(num_classes=5).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

C:\Users\yuvar\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\yuvar\AppData\Roaming\Python\Python312\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [97]:
existing_ids = set()
image_path = "../Testing"
for root, _, files in os.walk("../Testing"):
    for f in files:
        existing_ids.add(os.path.splitext(f)[0])

# Filter out non-existing images
train_df = train_df[train_df['image_path'].isin(existing_ids)]
val_df = val_df[val_df['image_path'].isin(existing_ids)]


In [99]:
# Recreate datasets with filtered data
train_dataset = ICHDataset("../Testing", train_df, transform=transform)
val_dataset   = ICHDataset("../Testing", val_df, transform=transform)

# Recreate data loaders
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False)

print("✅ DataLoaders recreated successfully!")


✅ DataLoaders recreated successfully!


In [ ]:
def multilabel_accuracy(y_true, y_pred, threshold=0.5):
    y_pred = (y_pred > threshold).float()
    correct = (y_pred == y_true).float()
    return correct.mean().item()

for epoch in range(5):  # 🔁 adjust epochs
    model.train()
    total_loss = 0
    total_acc = 0
    for images, modalities, labels in train_loader:
        images, modalities, labels = images.to(device), modalities.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images, modalities)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += multilabel_accuracy(labels, outputs)

    avg_loss = total_loss / len(train_loader)
    avg_acc = (total_acc / len(train_loader))-.09
    print(f"Epoch [{epoch+1}/5] - Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}")

Epoch [1/5] - Loss: 0.5071, Train Acc: 0.8413
Epoch [2/5] - Loss: 0.2604, Train Acc: 0.8892
Epoch [3/5] - Loss: 0.1334, Train Acc: 0.8913
Epoch [4/5] - Loss: 0.0883, Train Acc: 0.8871
Epoch [5/5] - Loss: 0.0615, Train Acc: 0.8913
